### Read chosen eval metric + image subset files for analysis 

5/11/2026

In [ ]:
from pathlib import Path

import base64
import html
import mimetypes
import pandas as pd
from IPython.display import HTML, display

repo_root = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "results" / "eval_results.csv").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Could not find results/eval_results.csv from the notebook directory"
    )

results_dir = repo_root / "results"
eval_csv = results_dir / "eval_results.csv"
eval_results = pd.read_csv(eval_csv)
if len(eval_results) != 1:
    raise ValueError(
        f"Expected one aggregate row in {eval_csv}, found {len(eval_results)}"
    )

print("Loaded pipeline averages from", eval_csv)
display(eval_results)


def show_metrics(*columns):
    missing = [column for column in columns if column not in eval_results.columns]
    if missing:
        raise KeyError(f"Missing metric columns in {eval_csv}: {missing}")
    display(eval_results.loc[:, columns].T.rename(columns={0: "average"}))


def show_result_images(*subdirectories):
    image_suffixes = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
    image_paths = []
    for subdirectory in subdirectories:
        image_dir = results_dir / subdirectory
        if image_dir.is_dir():
            image_paths.extend(
                sorted(
                    path
                    for path in image_dir.rglob("*")
                    if path.is_file() and path.suffix.lower() in image_suffixes
                )
            )

    if not image_paths:
        print("No images found in:", ", ".join(subdirectories))
        return

    print(f"Displaying {len(image_paths)} image(s) from:", ", ".join(subdirectories))
    for image_path in image_paths:
        mime_type = (
            mimetypes.guess_type(image_path.name)[0] or "application/octet-stream"
        )
        image_data = base64.b64encode(image_path.read_bytes()).decode("ascii")
        caption = html.escape(str(image_path.relative_to(results_dir)))
        display(
            HTML(
                f'<figure style="width:100%;max-width:100%;margin:1em 0;">'
                f'<img src="data:{mime_type};base64,{image_data}" '
                f'style="display:block;width:100%;max-width:100%;height:auto;">'
                f"<figcaption>{caption}</figcaption></figure>"
            )
        )

### Num of keypoints found 


In [ ]:
show_metrics("stage1_num_keypoints_mean")

### Num of matches per target keypoint


In [ ]:
show_metrics("stage1_avg_num_of_matches_per_target_keypoint_mean")

### Location of matches 


In [ ]:
show_result_images("feature_matches")

### Location of tube in relation to bounding box 


In [ ]:
show_result_images("tube_in_box")

### Location of edges 

In [ ]:
show_result_images("edge_detection")

### Number of hough lines total 

In [ ]:
show_metrics("stage2_num_hough_lines_total_mean")

### Num of hough lines in rough region 

In [ ]:
show_metrics("stage2_num_hough_lines_in_region_mean")

### Location of hough lines in rough region

In [ ]:
show_result_images("hough_lines")

### Num valid pairs found 

In [ ]:
show_metrics(
    "stage2_num_valid_pairs_vertical_mean",
    "stage2_num_valid_pairs_horizontal_mean",
)

### Location of valid pairs 


In [ ]:
show_result_images("valid_vertical_hough_lines", "valid_horizontal_hough_lines")

### Final similarity metrics to labelled ROI


In [ ]:
show_metrics("stage2_IoU_mean", "stage2_Dice_mean")

### Location of final ROI

In [ ]:
show_result_images("final_roi")